# Photometric ELG × spectroscopic references

Load the **separate BGS, LRG, ELG and QSO measurements and their combined result** from the DR1 run below. This notebook reads only small saved measurement tables, correlation-table headers and JSON/TSV metadata. It does not open survey FITS files, load trees, or run pair counting.

The quantity is the legacy code's locally normalized, scale-integrated correlation,
$\bar w(z)=\sum_i(\Delta r_i/r_i)w(r_i,z)$ over approximately $2.517$–$10\,h^{-1}\mathrm{Mpc}$, with leave-one-region-out jackknife errors. Conversion to $b\,dN/dz$ is still deferred. The photometric sample uses `elg_mask == 0`; photometric and reference weights are unity.

**Combined** is the direct measurement using the unique-reference catalog. Redshift-conflicting IDs are excluded from that catalog before binning, while the separate-tracer samples retain them. It is not an average of the four curves, and their errors are correlated. Different reference biases affect the amplitudes.

The orange histograms in the individual panels show the calibrators **actually used in saved measurements**, from the `N_SPEC` headers: $dN_{\rm ref}/dz=N_{\rm ref}/\Delta z$. They are not photometric ELG redshift distributions.

Use a kernel with NumPy and matplotlib (e.g. `elgcmb`). No imports from `scripts/` are needed.

In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import matplotlib.pyplot as plt

RUN_ROOT = Path('/pscratch/sd/e/ezaborow/elg-desi-dr1-full-60339137')
# An old SDSS run, a single tracer directory, or a single-bin pilot also works.
# RUN_ROOT = Path('/pscratch/sd/e/ezaborow/elg-sdss-full-60231840')
EXPECTED_EDGES = np.arange(81) / 20.  # Fallback for old runs without a manifest.
ZLIM = (0.0, 4.0)                    # Change to zoom, e.g. (0.4, 1.6).

TRACERS = [
    ('BGS_BRIGHT', 'bgs_bright', 'BGS', 'tab:blue'),
    ('LRG', 'lrg', 'LRG', 'tab:orange'),
    ('ELG_LOPnotqso', 'elg_lopnotqso', 'ELG', 'tab:green'),
    ('QSO', 'qso', 'QSO', 'tab:purple'),
    ('ALL', 'all', 'Combined', 'black'),
]

Re-run the loading and plotting cells to pick up new measurements. The status table uses `reference_counts.tsv` to distinguish **empty** bins from **missing** measurements. Invalid or partially written outputs are reported; no missing measurement is replaced with zero. The per-tracer arrays remain accessible as `results['LRG']['measurements']`, etc.; columns are `z_low, z_high, wbar, sigma_jackknife`.

In [ ]:
def load_measurements(directory):
    single = (directory / 'wbar.txt').is_file()
    files = [directory / 'wbar.txt'] if single else sorted(directory.glob('bin-[0-9][0-9][0-9]/wbar.txt'))
    rows, counts, sources, issues = [], [], [], []
    for path in files:
        try:
            table = np.loadtxt(path, ndmin=2)
            if table.shape[1] != 4 or not np.isfinite(table[:, :2]).all() or np.any(table[:, 1] <= table[:, 0]):
                raise ValueError('Expected four columns with valid redshift bounds')
        except (OSError, ValueError) as error:
            issues.append(f'{path}: {error}')
            continue
        for row in table:
            lo, hi, _, _ = row
            count = np.nan
            radial = path.parent / 'correlations' / f'z{lo:.2f}_{hi:.2f}_jk_loo_equalized.txt'
            if radial.is_file():
                with radial.open() as stream:
                    for line in stream:
                        if not line.startswith('#'):
                            break
                        if line.startswith('# N_SPEC='):
                            count = int(line.split('=', 1)[1])
                            break
            rows.append(row)
            counts.append(count)
            sources.append(str(path))
    measurements = np.asarray(rows, dtype=float).reshape(-1, 4)
    order = np.argsort(measurements[:, 0])
    measurements = measurements[order]
    counts = np.asarray(counts, dtype=float)[order]
    if np.any(measurements[1:, 0] < measurements[:-1, 1] - 1e-9):
        raise ValueError(f'Overlapping or duplicate bins in {directory}')
    valid = np.isfinite(measurements[:, 2:]).all(axis=1) & (measurements[:, 3] >= 0)
    widths = measurements[:, 1] - measurements[:, 0]
    return dict(measurements=measurements, n_reference=counts, valid=valid,
                reference_density=counts / widths, source_files=np.asarray(sources)[order],
                single=single, issues=issues)

if not RUN_ROOT.is_dir():
    raise FileNotFoundError(f'Run directory does not exist: {RUN_ROOT}')
multi_tracer = any((RUN_ROOT / folder).is_dir() for _, folder, _, _ in TRACERS)
if multi_tracer:
    datasets = [(key, RUN_ROOT / folder, label, color) for key, folder, label, color in TRACERS]
    manifest_root = RUN_ROOT
else:
    # Preserve support for the SDSS notebook's original layouts and DR1 pilots.
    info_path = RUN_ROOT / 'inputs' / 'selection.json'
    info = json.loads(info_path.read_text()) if info_path.is_file() else {}
    key = info.get('reference', {}).get('tracer', 'SDSS')
    for candidate, folder, _, _ in TRACERS:
        if RUN_ROOT.name == folder:
            key = candidate
    label, color = next(((label, color) for candidate, _, label, color in TRACERS if candidate == key), ('SDSS', 'tab:blue'))
    datasets = [(key, RUN_ROOT, label, color)]
    manifest_root = RUN_ROOT if (RUN_ROOT / 'reference_counts.tsv').is_file() else RUN_ROOT.parent

planned = {}
manifest = manifest_root / 'reference_counts.tsv'
if manifest.is_file():
    with manifest.open() as stream:
        for row in csv.DictReader(stream, delimiter='\t'):
            bounds = (round(float(row['z_low']), 8), round(float(row['z_high']), 8))
            planned.setdefault(row['tracer'], {})[bounds] = int(row['n_reference'])

results = {}
print(f'Run: {RUN_ROOT}\n')
print(f'{"Sample":<12} {"Saved":>6} {"Finite":>7} {"Invalid":>8} {"Missing":>8} {"Empty":>7}')
for key, directory, label, color in datasets:
    data = load_measurements(directory)
    bins = [tuple(np.round(row[:2], 8)) for row in data['measurements']]
    plan = planned.get(key)
    if plan is not None:
        expected = [bounds for bounds, count in plan.items() if count > 0]
        empty = sum(count == 0 for count in plan.values())
        for bounds, count in zip(bins, data['n_reference']):
            if bounds not in expected:
                raise ValueError(f'{label}: saved bin {bounds} is not in the nonempty task plan')
            if np.isfinite(count) and count != plan[bounds]:
                raise ValueError(f'{label} {bounds}: N_SPEC differs from the prepared reference count')
    else:
        expected = bins if data['single'] else [tuple(np.round(pair, 8)) for pair in zip(EXPECTED_EDGES[:-1], EXPECTED_EDGES[1:])]
        empty = None
    missing = [bounds for bounds in expected if bounds not in bins]
    data.update(label=label, color=color, missing=missing, expected=expected, empty=empty)
    results[key] = data
    finite = int(data['valid'].sum())
    print(f'{label:<12} {len(bins):6d} {finite:7d} {len(bins)-finite:8d} {len(missing):8d} {str(empty) if empty is not None else "—":>7}')
    if missing:
        print('  Missing:', ', '.join(f'{lo:.2f}–{hi:.2f}' for lo, hi in missing))
    for issue in data['issues']:
        print('  Unreadable/partial output:', issue)
    if (~data['valid']).any():
        print('  Invalid:', [bins[i] for i in np.flatnonzero(~data['valid'])])
    unknown = np.count_nonzero(~np.isfinite(data['n_reference']))
    if unknown:
        print(f'  Calibrator counts unavailable in {unknown} saved bins.')

selection_path = manifest_root / 'inputs' / 'selection.json'
if selection_path.is_file():
    reference = json.loads(selection_path.read_text()).get('reference', {})
    if reference.get('tracer') == 'ALL':
        print(f"\nGlobal combined catalog: {reference['selected_rows']:,} references; "
              f"{reference.get('excluded_redshift_targetids', 0):,} conflicting IDs excluded.")
print('\nEmpty = no references in the prepared catalog; missing = a planned bin without a saved measurement.')

In [ ]:
# Compare all available curves. Points are not connected across missing bins.
comparison_fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
for key, data in results.items():
    lo, hi, wbar, sigma = data['measurements'].T
    valid = data['valid']
    ax.errorbar((lo[valid] + hi[valid]) / 2, wbar[valid],
                xerr=(hi[valid] - lo[valid]) / 2, yerr=sigma[valid],
                fmt='s' if key == 'ALL' else 'o', markersize=4 if key == 'ALL' else 3,
                capsize=2, color=data['color'], alpha=1 if key == 'ALL' else .7,
                zorder=4 if key == 'ALL' else 3,
                label=f"{data['label']} ({valid.sum()} bins)")
ax.axhline(0, color='0.5', linestyle='--', linewidth=1)
ax.set(xlim=ZLIM, xlabel='Reference redshift', ylabel=r'$\bar w(z)$',
       title='Photometric ELG cross-correlations; unit weights; jackknife errors')
ax.legend(ncol=2)
ax.grid(alpha=.2)
ax.set_ylim(-0.025, 0.025)
plt.show()

# Each tracer, including Combined, with its own saved calibrator distribution.
ncols = 2 if len(results) > 1 else 1
nrows = (len(results) + ncols - 1) // ncols
detail_fig, axes = plt.subplots(nrows, ncols, figsize=(12 if ncols == 2 else 10, 3.8*nrows),
                                squeeze=False, constrained_layout=True)
for ax, (key, data) in zip(axes.flat, results.items()):
    lo, hi, wbar, sigma = data['measurements'].T
    centres, widths = (lo + hi) / 2, hi - lo
    valid = data['valid']
    counts = np.isfinite(data['n_reference']) & (data['n_reference'] >= 0)
    counts_ax = ax.twinx()
    counts_ax.bar(centres[counts], data['reference_density'][counts], width=widths[counts],
                  color='tab:orange', edgecolor='tab:orange', alpha=.25,
                  label='Calibrators (right axis)')
    counts_ax.set_ylim(bottom=0)
    counts_ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    counts_ax.set_ylabel(r'$dN_{\mathrm{ref}}/dz$', color='tab:orange')
    counts_ax.tick_params(axis='y', colors='tab:orange')
    ax.set_zorder(counts_ax.get_zorder() + 1)
    ax.patch.set_visible(False)
    ax.errorbar(centres[valid], wbar[valid], xerr=widths[valid]/2, yerr=sigma[valid],
                fmt='o', markersize=3, capsize=2, color=data['color'], label=r'$\bar w$ ± jackknife error')
    ax.axhline(0, color='0.5', linestyle='--', linewidth=1)
    ax.set(xlim=ZLIM, xlabel='Reference redshift', ylabel=r'$\bar w(z)$',
           title=f"{data['label']}: {valid.sum()}/{len(data['expected'])} finite bins")
    ax.grid(alpha=.2)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = counts_ax.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, fontsize=8, loc='best')
for ax in list(axes.flat)[len(results):]:
    ax.set_visible(False)
plt.show()

In [ ]:
# Optional detailed listing of the loaded small tables.
SHOW_TABLES = False
if SHOW_TABLES:
    for key, data in results.items():
        print(f"\n{data['label']}")
        print(f'{"z_low":>7} {"z_high":>7} {"wbar":>13} {"sigma_jk":>13} {"N_ref":>9} {"status":>8}')
        for row, nref, ok in zip(data['measurements'], data['n_reference'], data['valid']):
            lo, hi, value, error = row
            count = str(int(nref)) if np.isfinite(nref) else 'unknown'
            print(f'{lo:7.2f} {hi:7.2f} {value:13.6g} {error:13.6g} {count:>9} {"finite" if ok else "invalid":>8}')

The curves share photometric objects and jackknife regions. These files provide per-bin errors, **not** covariance between tracers or redshift bins. Inspect missing and invalid bins before interpreting the full measurement.

Exports below are disabled by default. Enabling them saves both figures and separate measurement/reference-count tables for every sample, and refuses to overwrite an existing export directory.

In [ ]:
EXPORT = False
EXPORT_DIR = Path.cwd() / 'elg_desi_dr1_plot_export'
if EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=False)
    for key, data in results.items():
        prefix = key.lower()
        np.savetxt(EXPORT_DIR / f'{prefix}_wbar_vs_z.txt', data['measurements'],
                   header='z_low z_high wbar sigma_jackknife; saved bins only')
        np.savetxt(EXPORT_DIR / f'{prefix}_reference_dndz.txt',
                   np.column_stack((data['measurements'][:, :2], data['n_reference'], data['reference_density'])),
                   header='z_low z_high N_ref dN_ref_dz; saved bins only; unknown counts are NaN')
    for name, fig in [('comparison', comparison_fig), ('per_tracer', detail_fig)]:
        fig.savefig(EXPORT_DIR / f'{name}.png', dpi=180)
        fig.savefig(EXPORT_DIR / f'{name}.pdf')
    print(f'Exported to {EXPORT_DIR.resolve()}')

In [ ]:
# Compare all available curves. Points are not connected across missing bins.
comparison_fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
for key, data in results.items():
    lo, hi, wbar, sigma = data['measurements'].T
    valid = data['valid']
    ax.errorbar((lo[valid] + hi[valid]) / 2, wbar[valid],
                xerr=(hi[valid] - lo[valid]) / 2, yerr=sigma[valid],
                fmt='s' if key == 'ALL' else 'o', markersize=4 if key == 'ALL' else 3,
                capsize=2, color=data['color'], alpha=1 if key == 'ALL' else .7,
                zorder=4 if key == 'ALL' else 3,
                label=f"{data['label']} ({valid.sum()} bins)")
ax.axhline(0, color='0.5', linestyle='--', linewidth=1)
ax.set(xlim=ZLIM, xlabel='Reference redshift', ylabel=r'$\bar w(z)$',
       title='Photometric ELG cross-correlations; unit weights; jackknife errors')
ax.legend(ncol=2)
ax.grid(alpha=.2)
plt.show()

# Each tracer, including Combined, with its own saved calibrator distribution.
ncols = 2 if len(results) > 1 else 1
nrows = (len(results) + ncols - 1) // ncols
detail_fig, axes = plt.subplots(nrows, ncols, figsize=(12 if ncols == 2 else 10, 3.8*nrows),
                                squeeze=False, constrained_layout=True)
for ax, (key, data) in zip(axes.flat, results.items()):
    lo, hi, wbar, sigma = data['measurements'].T
    centres, widths = (lo + hi) / 2, hi - lo
    valid = data['valid']
    counts = np.isfinite(data['n_reference']) & (data['n_reference'] >= 0)
    counts_ax = ax.twinx()
    counts_ax.bar(centres[counts], data['reference_density'][counts], width=widths[counts],
                  color='tab:orange', edgecolor='tab:orange', alpha=.25,
                  label='Calibrators (right axis)')
    counts_ax.set_ylim(bottom=0)
    counts_ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    counts_ax.set_ylabel(r'$dN_{\mathrm{ref}}/dz$', color='tab:orange')
    counts_ax.tick_params(axis='y', colors='tab:orange')
    ax.set_zorder(counts_ax.get_zorder() + 1)
    ax.patch.set_visible(False)
    ax.errorbar(centres[valid], wbar[valid], xerr=widths[valid]/2, yerr=sigma[valid],
                fmt='o', markersize=3, capsize=2, color=data['color'], label=r'$\bar w$ ± jackknife error')
    ax.axhline(0, color='0.5', linestyle='--', linewidth=1)
    ax.set(xlim=ZLIM, xlabel='Reference redshift', ylabel=r'$\bar w(z)$',
           title=f"{data['label']}: {valid.sum()}/{len(data['expected'])} finite bins")
    ax.grid(alpha=.2)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = counts_ax.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, fontsize=8, loc='best')
for ax in list(axes.flat)[len(results):]:
    ax.set_visible(False)
plt.show()

In [ ]:
# Compare all available curves. Points are not connected across missing bins.
comparison_fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
for key, data in results.items():
    lo, hi, wbar, sigma = data['measurements'].T
    valid = data['valid']
    ax.errorbar((lo[valid] + hi[valid]) / 2, wbar[valid],
                xerr=(hi[valid] - lo[valid]) / 2, yerr=sigma[valid],
                fmt='s' if key == 'ALL' else 'o', markersize=4 if key == 'ALL' else 3,
                capsize=2, color=data['color'], alpha=1 if key == 'ALL' else .7,
                zorder=4 if key == 'ALL' else 3,
                label=f"{data['label']} ({valid.sum()} bins)")
ax.axhline(0, color='0.5', linestyle='--', linewidth=1)
ax.set(xlim=ZLIM, xlabel='Reference redshift', ylabel=r'$\bar w(z)$',
       title='Photometric ELG cross-correlations; unit weights; jackknife errors')
ax.legend(ncol=2)
ax.grid(alpha=.2)
ax.set_ylim(-0.025, 0.025)
plt.show()

# Each tracer, including Combined, with its own saved calibrator distribution.
ncols = 2 if len(results) > 1 else 1
nrows = (len(results) + ncols - 1) // ncols
detail_fig, axes = plt.subplots(nrows, ncols, figsize=(12 if ncols == 2 else 10, 3.8*nrows),
                                squeeze=False, constrained_layout=True)
for ax, (key, data) in zip(axes.flat, results.items()):
    lo, hi, wbar, sigma = data['measurements'].T
    centres, widths = (lo + hi) / 2, hi - lo
    valid = data['valid']
    counts = np.isfinite(data['n_reference']) & (data['n_reference'] >= 0)
    counts_ax = ax.twinx()
    counts_ax.bar(centres[counts], data['reference_density'][counts], width=widths[counts],
                  color='tab:orange', edgecolor='tab:orange', alpha=.25,
                  label='Calibrators (right axis)')
    counts_ax.set_ylim(bottom=0)
    counts_ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    counts_ax.set_ylabel(r'$dN_{\mathrm{ref}}/dz$', color='tab:orange')
    counts_ax.tick_params(axis='y', colors='tab:orange')
    ax.set_zorder(counts_ax.get_zorder() + 1)
    ax.patch.set_visible(False)
    ax.errorbar(centres[valid], wbar[valid], xerr=widths[valid]/2, yerr=sigma[valid],
                fmt='o', markersize=3, capsize=2, color=data['color'], label=r'$\bar w$ ± jackknife error')
    ax.axhline(0, color='0.5', linestyle='--', linewidth=1)
    ax.set(xlim=ZLIM, xlabel='Reference redshift', ylabel=r'$\bar w(z)$',
           title=f"{data['label']}: {valid.sum()}/{len(data['expected'])} finite bins")
    ax.grid(alpha=.2)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = counts_ax.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, fontsize=8, loc='best')
for ax in list(axes.flat)[len(results):]:
    ax.set_visible(False)
plt.show()